In [205]:
import pandas as pd
from astroquery.gaia import Gaia
import numpy as np

In [206]:
### download sed fitting results 8/04/26
raw_sed = pd.read_csv('/Users/z5214708/Desktop/stage_4/SED_results_kbonus_extra.dat', sep='\s+')
names = raw_sed.columns[1:]
sed_results = pd.read_csv('/Users/z5214708/Desktop/stage_4/SED_results_kbonus_extra.dat', sep='\s+', header=None, skiprows=1, names=names)

##################################

In [207]:
#format columns
sed_results = sed_results[['source_id','Teff','s_Teff','feh','s_feh']].copy()

In [208]:
#luca said he didnt rerun the ones he already did- check which ones are missing from the list we sent him
original_file = pd.read_csv('/Users/z5214708/Desktop/stage_4/kbonus_sample.csv')

missing = original_file[~original_file['source_id'].isin(sed_results['source_id'])].copy()

In [209]:
# Read the first SED results (these are the ones where logg = 2.5)
first_sed_raw = pd.read_csv('/Users/z5214708/Desktop/SED/res25_sf11_sed', sep='\s+')
first_names = first_sed_raw.columns[1:]
first_sed_results = pd.read_csv('/Users/z5214708/Desktop/SED/res25_sf11_sed', sep='\s+', header=None, skiprows=1, names=first_names)
#format to match new sed results
first_sed_results = first_sed_results[['source_id', 'Teff', 'eTeff', 'feh', 'e_feh']].copy()
first_sed_results.rename(columns={'eTeff': 's_Teff', 'e_feh': 's_feh'}, inplace=True)

extras = pd.read_csv('/Users/z5214708/Desktop/SEDonly_results_kbonus.dat',sep='\s+')
extras_names = extras.columns[1:]
extras = pd.read_csv('/Users/z5214708/Desktop/SEDonly_results_kbonus.dat', sep='\s+', header=None, skiprows=1, names=extras_names)
extras = extras[['source_id','Teff','s_Teff','feh','s_feh']].copy()

#add together
first_sed_results = pd.concat([first_sed_results,extras], ignore_index=True)


In [210]:
first_sed_results = first_sed_results[first_sed_results['source_id'].isin(missing['source_id'])]
 #only keep stars that are in the current kbonus sanple

In [211]:
sed_results = pd.concat([sed_results, first_sed_results], ignore_index=True)

In [212]:
#add floor to uncertainties
sed_results = sed_results[sed_results['Teff'] > 0].copy()#get rid of two stars with negative results
sed_results['s_feh'] = sed_results['s_feh'] + 0.2
sed_results['s_Teff'] = sed_results['s_Teff'] + 100

In [213]:
#get gaia values
source_ids =original_file['source_id']
batch_size=1000
all_results = []
for i in range(0, len(source_ids), batch_size):
    batch = source_ids[i:i + batch_size]  
    id_string = ','.join(map(str, batch))
        
    query = f"""
        SELECT source_id, ra, dec, phot_g_mean_mag, phot_g_mean_flux, phot_g_mean_flux_error, parallax, parallax_error
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({id_string})
        """
        
    job = Gaia.launch_job(query)
    all_results.append(job.get_results().to_pandas())
all_results = pd.concat(all_results)

#merge to get gaia values
missing_output = missing.merge(all_results, on='source_id', how='left')
sed_output = sed_results.merge(all_results, on='source_id', how='left')

In [214]:
#save results
missing_output.to_csv('../data/missing_sed_results.csv',index=False)
sed_output.to_csv('../data/sed_results.csv',index=False)